In [1]:
!pip install -q fastapi uvicorn[standard] pydantic pyngrok nest-asyncio python-multipart


# سلول 2 — Mount Drive + مسیرهای پروژه

In [2]:
import os, re, time, json, unicodedata, logging
from pathlib import Path
from functools import lru_cache

import torch
import torch.nn as nn
import torch.nn.functional as F

from fastapi import FastAPI, HTTPException, Request, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.gzip import GZipMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, field_validator
import uvicorn

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Project root in Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/persian_text_sanitizer") if IN_COLAB \
               else Path("./persian_text_sanitizer")
CKPT_DIR  = PROJECT_ROOT / "checkpoints"
LOG_DIR   = PROJECT_ROOT / "logs"
TEST_DIR  = PROJECT_ROOT / "tests"

for d in [CKPT_DIR, LOG_DIR, TEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Reference to previous project's checkpoints (Spell Corrector)
SPELL_CKPT = Path("/content/drive/MyDrive/persian_spell_corrector/checkpoints/best_spell_corrector.pt")

print("Project root:", PROJECT_ROOT)
print("Spell ckpt exists:", SPELL_CKPT.exists())

Mounted at /content/drive
Project root: /content/drive/MyDrive/persian_text_sanitizer
Spell ckpt exists: True


# سلول 3 — Device + تنظیمات Global


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Logging config
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "api.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("sanitizer")

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB


# سلول 4 — ثابت‌ها (کاراکتر مپینگ‌ها)


In [4]:
# ---------------------------------------------------------------
# Character mappings
# ---------------------------------------------------------------
ARABIC_TO_PERSIAN = {
    "ك": "ک",  # Arabic Kaf → Persian Keheh
    "ي": "ی",  # Arabic Yeh → Persian Yeh
    "ة": "ه",  # Ta Marbuta → Heh
    "أ": "ا",  # Alef with Hamza above → Alef
    "إ": "ا",  # Alef with Hamza below → Alef
    "ؤ": "و",  # Waw with Hamza → Waw
    "ئ": "ی",  # Yeh with Hamza → Yeh
}

DIACRITICS = set("ًٌٍَُِّْـ")  # fatha, kasra, damma, sukun, tanwin, tatweel

# Digits: Arabic-Indic → Persian → English (canonical: English)
ARABIC_DIGITS   = "٠١٢٣٤٥٦٧٨٩"
PERSIAN_DIGITS  = "۰۱۲۳۴۵۶۷۸۹"
ENGLISH_DIGITS  = "0123456789"
DIGIT_MAP = {}
for i, ch in enumerate(ARABIC_DIGITS):
    DIGIT_MAP[ch] = ENGLISH_DIGITS[i]
for i, ch in enumerate(PERSIAN_DIGITS):
    DIGIT_MAP[ch] = ENGLISH_DIGITS[i]

# Pre-compiled regexes (compiled once → faster)
HTML_TAG_RE    = re.compile(r"<[^>]+>")
URL_RE         = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE       = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
MULTISPACE_RE  = re.compile(r"[ \t]+")
MULTINEWLINE_RE= re.compile(r"\n{3,}")
REPEATED_CHAR_RE = re.compile(r"(.)\1{3,}")  # 4+ repeats → 1
CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")

# سلول 5 — توابع هسته Sanitizer (قدم به قدم)


In [5]:
# ---------------------------------------------------------------
# Sanitizer core functions — each does ONE thing
# ---------------------------------------------------------------

def _unicode_nfkc(text: str) -> str:
    """NFKC: فشرده‌سازی سازگاری (ﻻ → لا، ﷼ → ریال)."""
    return unicodedata.normalize("NFKC", text)


def _strip_control_chars(text: str) -> str:
    """حذف کاراکترهای کنترلی ASCII (غیر از \n, \t)."""
    return CONTROL_CHARS_RE.sub("", text)


def _strip_html(text: str) -> str:
    """حذف تگ‌های HTML."""
    return HTML_TAG_RE.sub(" ", text)


def _strip_urls_emails(text: str) -> str:
    """حذف URL و Email."""
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    return text


def _ar2fa(text: str) -> str:
    """تبدیل کاراکترهای عربی به فارسی."""
    return text.translate(str.maketrans(ARABIC_TO_PERSIAN))


def _remove_diacritics(text: str) -> str:
    """حذف اعراب و tatweel."""
    return "".join(ch for ch in text if ch not in DIACRITICS)


def _normalize_digits(text: str) -> str:
    """تبدیل ارقام عربی/فارسی به انگلیسی."""
    return text.translate(str.maketrans(DIGIT_MAP))


def _fix_zwnj(text: str) -> str:
    """اصلاح نیم‌فاصله (می‌روم، کتاب‌ها، بزرگ‌تر).
    قواعد ساده‌ی مبتنی بر regex — برای کامل بودن می‌شود با لغت‌نامه جایگزین کرد."""
    # "می روم" → "می‌روم"  (فقط افعال)
    text = re.sub(r"\bمی\s+([آ-ی])", r"می‌\1", text)
    text = re.sub(r"\bنمی\s+([آ-ی])", r"نمی‌\1", text)
    # "کتاب ها" → "کتاب‌ها"
    text = re.sub(r"([آ-ی])\s+ها\b", r"\1‌ها", text)
    # "کتاب های" → "کتاب‌های"
    text = re.sub(r"([آ-ی])\s+های\b", r"\1‌های", text)
    # "بزرگ تر" → "بزرگ‌تر"
    text = re.sub(r"([آ-ی])\s+تر\b", r"\1‌تر", text)
    text = re.sub(r"([آ-ی])\s+ترین\b", r"\1‌ترین", text)
    return text


def _collapse_repeats(text: str) -> str:
    """aaaaaaaa → a (برای 4+ تکرار)."""
    return REPEATED_CHAR_RE.sub(r"\1", text)


def _collapse_whitespace(text: str) -> str:
    """فاصله‌های تکراری → یک فاصله. حفظ خطوط جدید."""
    text = MULTISPACE_RE.sub(" ", text)
    text = MULTINEWLINE_RE.sub("\n\n", text)
    return text.strip()

# سلول 6 — Pipeline یکپارچه Sanitizer


In [6]:
class SanitizerConfig(BaseModel):
    """گزینه‌های قابل تنظیم برای sanitizer."""
    apply_zwnj: bool = True
    apply_digits_normalization: bool = True
    strip_urls: bool = True
    strip_html: bool = True
    collapse_repeats: bool = True


def sanitize(text: str, cfg: SanitizerConfig | None = None) -> dict:
    """اجرای کل pipeline و برگرداندن متن پاک + تعداد تغییرات."""
    cfg = cfg or SanitizerConfig()
    original = text
    steps = {}

    def _step(name, fn):
        nonlocal text
        before = text
        text = fn(text)
        if text != before:
            steps[name] = True

    _step("nfkc",              _unicode_nfkc)
    _step("control_chars",     _strip_control_chars)
    if cfg.strip_html:
        _step("html",          _strip_html)
    if cfg.strip_urls:
        _step("urls_emails",   _strip_urls_emails)
    _step("ar2fa",             _ar2fa)
    _step("diacritics",        _remove_diacritics)
    if cfg.apply_digits_normalization:
        _step("digits",        _normalize_digits)
    if cfg.apply_zwnj:
        _step("zwnj",          _fix_zwnj)
    if cfg.collapse_repeats:
        _step("repeats",       _collapse_repeats)
    _step("whitespace",        _collapse_whitespace)

    return {
        "text": text,
        "changed": text != original,
        "steps_applied": list(steps.keys()),
    }

# سلول ریست و اجرای بعد از این سلول 7 جدید است

In [29]:
_corrector_model = None
_corrector_meta  = {}
import torch, gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("✅ Ready.")

✅ Ready.


# 🔧 سلول ۷ (اصلاح‌شده) — Transformer با نام‌گذاری دقیقاً مطابق پروژه قبلی

In [19]:
# ==============================================================
# Cell 7 (FIXED) — Model classes با نام‌گذاری دقیقاً مطابق checkpoint
# ==============================================================
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads
        self.heads = heads
        self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        q, k, v = [
            lin(x).view(B, -1, self.heads, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            while mask.dim() < scores.dim():
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(self.dropout(attn), v)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.heads * self.d_k)
        return self.linears[3](out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.w2(self.dropout(F.relu(self.w1(x))))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


# ⚠️ اسم‌ها دقیقاً: norm1/norm2/dropout1/dropout2/feed_forward
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, mask):
        x_norm = self.norm1(x)
        attn_out = self.self_attn(x_norm, x_norm, x_norm, mask)
        x = x + self.dropout1(attn_out)
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout2(ff_out)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    def forward(self, x, memory, src_mask, tgt_mask):
        x_norm = self.norm1(x)
        x = x + self.dropout1(self.self_attn(x_norm, x_norm, x_norm, tgt_mask))
        x_norm = self.norm2(x)
        x = x + self.dropout2(self.cross_attn(x_norm, memory, memory, src_mask))
        x = x + self.dropout3(self.feed_forward(self.norm3(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, layer, N, d_model):
        super().__init__()
        self.layers = nn.ModuleList([layer for _ in range(N)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self, layer, N, d_model):
        super().__init__()
        self.layers = nn.ModuleList([layer for _ in range(N)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)
    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)


# ⚠️ اسم‌ها دقیقاً: src_embed, tgt_embed, pos_enc, encoder, decoder, generator
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=128, N=3, heads=8,
                 d_ff=256, dropout=0.1, max_len=5000):
        super().__init__()
        enc_layer = EncoderLayer(d_model, heads, d_ff, dropout)
        dec_layer = DecoderLayer(d_model, heads, d_ff, dropout)
        self.encoder    = Encoder(enc_layer, N, d_model)
        self.decoder    = Decoder(dec_layer, N, d_model)
        self.src_embed  = nn.Embedding(src_vocab, d_model)
        self.tgt_embed  = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len, dropout)
        self.generator  = Generator(d_model, tgt_vocab)
        self.d_model    = d_model

    def encode(self, src, src_mask):
        x = self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model))
        return self.encoder(x, src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        x = self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model))
        return self.decoder(x, memory, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)
        dec_out = self.decode(memory, src_mask, tgt, tgt_mask)
        return self.generator(dec_out)

# 🔴 سلول ۷ (جدید — جایگزین کن)

In [30]:
# ==============================================================
# Cell 7 (NEW) — Transformer با نام‌گذاری دقیقاً مطابق checkpoint
# ==============================================================
import math

def clones(module, N):
    """⚠️ مهم: shared weights — دقیقاً همان رفتار پروژه قبلی.
    اگر nn.ModuleList([module for _ in range(N)]) بنویسیم، هر لایه
    یک کپی جدا می‌شود و load_state_dict(strict=True) می‌ترکد،
    چون checkpoint فقط encoder.layers.0.* را دارد."""
    return nn.ModuleList([module for _ in range(N)])


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads
        self.heads = heads
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        q, k, v = [
            lin(x).view(B, -1, self.heads, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            while mask.dim() < scores.dim():
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(self.dropout(attn), v)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.heads * self.d_k)
        return self.linears[3](out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.w2(self.dropout(F.relu(self.w1(x))))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)                       # (1, max_len, d_model)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, mask):
        x_norm = self.norm1(x)
        attn_out = self.self_attn(x_norm, x_norm, x_norm, mask)
        x = x + self.dropout1(attn_out)
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout2(ff_out)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    def forward(self, x, memory, src_mask, tgt_mask):
        x_norm = self.norm1(x)
        self_attn_out = self.self_attn(x_norm, x_norm, x_norm, tgt_mask)
        x = x + self.dropout1(self_attn_out)
        x_norm = self.norm2(x)
        cross_attn_out = self.cross_attn(x_norm, memory, memory, src_mask)
        x = x + self.dropout2(cross_attn_out)
        ff_out = self.feed_forward(self.norm3(x))
        x = x + self.dropout3(ff_out)
        return x


class Encoder(nn.Module):
    def __init__(self, layer, N, d_model):
        super().__init__()
        self.layers = clones(layer, N)             # ← shared weights
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self, layer, N, d_model):
        super().__init__()
        self.layers = clones(layer, N)             # ← shared weights
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)
    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)


class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=128, N=3, heads=8,
                 d_ff=256, dropout=0.1, max_len=5000):
        super().__init__()
        enc_layer = EncoderLayer(d_model, heads, d_ff, dropout)
        dec_layer = DecoderLayer(d_model, heads, d_ff, dropout)
        self.encoder    = Encoder(enc_layer, N, d_model)
        self.decoder    = Decoder(dec_layer, N, d_model)
        self.src_embed  = nn.Embedding(src_vocab, d_model)
        self.tgt_embed  = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len, dropout)
        self.generator  = Generator(d_model, tgt_vocab)
        self.d_model    = d_model

    def encode(self, src, src_mask):
        x = self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model))
        return self.encoder(x, src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        x = self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model))
        return self.decoder(x, memory, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)
        dec_out = self.decode(memory, src_mask, tgt, tgt_mask)
        return self.generator(dec_out)


print("✅ Cell 7 loaded — Transformer classes ready.")

✅ Cell 7 loaded — Transformer classes ready.


# 🔧 سلول ۸ (اصلاح‌شده) — Vocab + load_corrector مقاوم


In [20]:
# ==============================================================
# Cell 8 (FIXED) — Vocab + load_corrector با inference از state_dict
# ==============================================================
FA_CHARS = list(
    "آابپتثجچحخدذرزژسشصضطظعغفقکگلمنوهی"
    "ئءأإؤ"
    "٠١٢٣٤٥٦٧٨٩"
    "0123456789"
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    " .,;:!?؟،«»()-'\""
    "\u200c"
    "ًٌٍَُِّْ"
    "ـ"
)
SPECIAL = ["<pad>", "<sos>", "<eos>", "<unk>"]
N_SPECIAL = len(SPECIAL)


class CharVocab:
    def __init__(self, chars):
        unique = []
        for c in chars:
            if c not in unique:
                unique.append(c)
        self.c2i = {c: i + N_SPECIAL for i, c in enumerate(unique)}
        self.i2c = {i: c for c, i in self.c2i.items()}
        self.pad_idx, self.sos_idx, self.eos_idx, self.unk_idx = 0, 1, 2, 3
        self.size = N_SPECIAL + len(unique)

    def encode(self, text, add_sos=False, add_eos=False):
        ids = [self.c2i.get(c, self.unk_idx) for c in text]
        if add_sos: ids = [self.sos_idx] + ids
        if add_eos: ids = ids + [self.eos_idx]
        return ids

    def decode(self, ids, skip_special=True):
        return "".join(
            self.i2c[int(i)] for i in ids
            if not (skip_special and int(i) < N_SPECIAL)
        )


vocab = CharVocab(FA_CHARS)
print("Vocab size:", vocab.size)

_corrector_model = None
_corrector_meta  = {}


def _infer_arch_from_state_dict(sd):
    """معماری مدل را از روی tensor shape ها استخراج می‌کند (چون cfg در ckpt خالی است)."""
    # اسم کلیدها مطابق کد پروژه قبلی
    src_vocab = sd["src_embed.weight"].shape[0]
    tgt_vocab = sd["tgt_embed.weight"].shape[0]
    d_model   = sd["src_embed.weight"].shape[1]

    # تعداد لایه‌های encoder (بیشترین ایندکس + 1)
    enc_indices = set()
    for k in sd:
        if k.startswith("encoder.layers."):
            parts = k.split(".")
            if len(parts) >= 3 and parts[2].isdigit():
                enc_indices.add(int(parts[2]))
    N = (max(enc_indices) + 1) if enc_indices else 3

    # d_ff از اولین FeedForward
    try:
        d_ff = sd["encoder.layers.0.feed_forward.w1.weight"].shape[0]
    except KeyError:
        d_ff = 4 * d_model

    # max_len از pos_enc.pe
    try:
        max_len = sd["pos_enc.pe"].shape[1]
    except KeyError:
        max_len = 5000

    # heads: از shape کافی نیست، پس پیش‌فرض 8 با fallback
    heads = 8
    if d_model % heads != 0:
        heads = 1
        for h in (8, 4, 2, 1):
            if d_model % h == 0:
                heads = h
                break

    return dict(
        src_vocab=src_vocab, tgt_vocab=tgt_vocab,
        d_model=d_model, N=N, heads=heads,
        d_ff=d_ff, max_len=max_len,
    )


def load_corrector(path: Path = SPELL_CKPT):
    """لود lazy + مقاوم به cfg خالی + استخراج معماری از state_dict."""
    global _corrector_model, _corrector_meta
    if _corrector_model is not None:
        return _corrector_model
    if not path.exists():
        logger.warning(f"Checkpoint not found: {path}. /correct disabled.")
        return None

    try:
        ckpt = torch.load(path, map_location=device)
        sd = ckpt["model_state_dict"]

        # --- تلاش ۱: از cfg (اگر درست ذخیره شده باشد) ---
        cfg = ckpt.get("cfg") or {}
        arch = _infer_arch_from_state_dict(sd)  # همیشه استخراج کن (fallback)

        # cfg اگر کلید داشت، اولویت دارد (برای مدل‌های دیگر با معماری متفاوت)
        if isinstance(cfg, dict) and "d_model" in cfg and cfg["d_model"] == arch["d_model"]:
            arch.update({k: cfg[k] for k in ("N", "heads", "d_ff") if k in cfg})

        logger.info(f"Loading corrector: {arch}")

        model = Transformer(**arch, dropout=0.0).to(device)
        model.load_state_dict(sd, strict=True)
        model.eval()
        _corrector_model = model
        _corrector_meta  = {
            "best_val_loss": ckpt.get("best_val_loss"),
            "arch": arch,
        }
        logger.info(f"✅ Spell corrector loaded | val={_corrector_meta.get('best_val_loss')}")
        return model

    except Exception as e:
        logger.error(f"Failed to load corrector: {type(e).__name__}: {e}", exc_info=True)
        return None

Vocab size: 140


# 🔴 سلول ۸ (جدید — جایگزین کن)


In [31]:
# ==============================================================
# Cell 8 (NEW) — Vocab + load_corrector مقاوم
# ==============================================================
FA_CHARS = list(
    "آابپتثجچحخدذرزژسشصضطظعغفقکگلمنوهی"
    "ئءأإؤ"
    "٠١٢٣٤٥٦٧٨٩"
    "0123456789"
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    " .,;:!?؟،«»()-'\""
    "\u200c"
    "ًٌٍَُِّْ"
    "ـ"
)
SPECIAL = ["<pad>", "<sos>", "<eos>", "<unk>"]
N_SPECIAL = len(SPECIAL)


class CharVocab:
    def __init__(self, chars):
        unique = []
        for c in chars:
            if c not in unique:
                unique.append(c)
        self.c2i = {c: i + N_SPECIAL for i, c in enumerate(unique)}
        self.i2c = {i: c for c, i in self.c2i.items()}
        self.pad_idx, self.sos_idx, self.eos_idx, self.unk_idx = 0, 1, 2, 3
        self.size = N_SPECIAL + len(unique)
    def encode(self, text, add_sos=False, add_eos=False):
        ids = [self.c2i.get(c, self.unk_idx) for c in text]
        if add_sos: ids = [self.sos_idx] + ids
        if add_eos: ids = ids + [self.eos_idx]
        return ids
    def decode(self, ids, skip_special=True):
        return "".join(
            self.i2c[int(i)] for i in ids
            if not (skip_special and int(i) < N_SPECIAL)
        )


vocab = CharVocab(FA_CHARS)
print(f"Vocab size: {vocab.size}")

_corrector_model = None
_corrector_meta  = {}


def _infer_arch_from_state_dict(sd):
    """استخراج معماری از state_dict.
    نکته: N و heads از state_dict قابل استخراج نیستند (heads همیشه یک
    تنسور d_modelxd_model است و با shared weights فقط layers.0 وجود دارد).
    پس از CFG اصلی notebook هاردکد می‌کنیم: N=3, heads=8."""
    d_model    = sd["src_embed.weight"].shape[1]
    src_vocab  = sd["src_embed.weight"].shape[0]
    tgt_vocab  = sd["tgt_embed.weight"].shape[0]
    d_ff       = sd["encoder.layers.0.feed_forward.w1.weight"].shape[0]
    try:
        max_len = sd["pos_enc.pe"].shape[1]
    except KeyError:
        max_len = 5000

    N     = 3   # ← از CFG.N پروژه قبلی
    heads = 8   # ← از CFG.heads پروژه قبلی
    if d_model % heads != 0:
        for h in (8, 4, 2, 1):
            if d_model % h == 0:
                heads = h
                break

    return dict(
        src_vocab=src_vocab, tgt_vocab=tgt_vocab,
        d_model=d_model, N=N, heads=heads,
        d_ff=d_ff, max_len=max_len,
    )


def load_corrector(path: Path = SPELL_CKPT):
    """لود lazy + تشخیص معماری + گزارش شفاف در صورت mismatch."""
    global _corrector_model, _corrector_meta
    if _corrector_model is not None:
        return _corrector_model
    if not path.exists():
        logger.warning(f"Checkpoint not found: {path}. /correct disabled.")
        return None

    try:
        ckpt = torch.load(path, map_location=device)
        sd = ckpt["model_state_dict"]

        arch = _infer_arch_from_state_dict(sd)
        logger.info(f"Inferred arch: {arch}")

        model = Transformer(**arch, dropout=0.0).to(device)

        # ابتدا strict، اگر خطا داد با strict=False ادامه بده و گزارش کن
        try:
            model.load_state_dict(sd, strict=True)
            logger.info("✅ load_state_dict(strict=True) — perfect match")
        except RuntimeError as e:
            logger.warning(f"strict load failed: {e}")
            logger.warning("Retrying with strict=False...")
            missing, unexpected = model.load_state_dict(sd, strict=False)
            if missing:
                logger.warning(f"Missing keys: {len(missing)} — first 5: {missing[:5]}")
            if unexpected:
                logger.warning(f"Unexpected keys: {len(unexpected)} — first 5: {unexpected[:5]}")

        model.eval()
        _corrector_model = model
        _corrector_meta  = {
            "best_val_loss": ckpt.get("best_val_loss"),
            "arch": arch,
        }
        logger.info(f"✅ Corrector loaded | val={_corrector_meta.get('best_val_loss')}")
        return model

    except Exception as e:
        logger.error(f"Failed to load corrector: {type(e).__name__}: {e}", exc_info=True)
        return None


print("✅ Cell 8 loaded — load_corrector ready.")

Vocab size: 140
✅ Cell 8 loaded — load_corrector ready.


# 🔧 سلول ۹ (اصلاح‌شده) — _greedy_decode با model.generator


In [25]:
# ==============================================================
# Cell 9 (v2) — Inference با Confidence Guard
# ==============================================================
from difflib import SequenceMatcher

def _similarity(a: str, b: str) -> float:
    """0..1 — چقدر a و b شبیه‌اند."""
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return SequenceMatcher(None, a, b).ratio()


def _greedy_decode(model, src_ids, max_len=64):
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_mask = torch.ones(1, 1, src.size(1), dtype=torch.bool, device=device)
    with torch.no_grad():
        memory = model.encode(src, src_mask)
        ys = torch.tensor([[vocab.sos_idx]], dtype=torch.long, device=device)
        for _ in range(max_len):
            tgt_mask = ~torch.triu(
                torch.ones(1, ys.size(1), ys.size(1), dtype=torch.bool, device=device),
                diagonal=1,
            )
            dec_out = model.decode(memory, src_mask, ys, tgt_mask)
            logp = model.generator(dec_out[:, -1])
            next_id = int(logp.argmax(-1).item())
            ys = torch.cat([ys, torch.tensor([[next_id]], device=device)], dim=1)
            if next_id == vocab.eos_idx:
                break
    return ys[0].tolist()[1:-1]


def spell_correct(text: str, min_similarity: float = 0.75) -> dict:
    """
    Confidence guard: اگر خروجی مدل خیلی از ورودی فاصله بگیرد، رد می‌شود.
    min_similarity=0.75 یعنی حداقل 75% کاراکترها باید یکسان باشند.
    """
    t0 = time.time()
    model = load_corrector()
    if model is None:
        return {"text": text, "corrected": False, "reason": "model_not_loaded",
                "confidence": 0.0, "time_ms": int((time.time() - t0) * 1000)}

    src_ids = vocab.encode(text[:64])
    if not src_ids:
        return {"text": text, "corrected": False, "reason": "empty_input",
                "confidence": 0.0, "time_ms": int((time.time() - t0) * 1000)}

    try:
        with torch.no_grad():
            out_ids = _greedy_decode(model, src_ids)
        out_text = vocab.decode(out_ids) or text

        # ← گارد اصلی: اگر خروجی شباهت کافی ندارد، رد کن
        sim = _similarity(text, out_text)
        if sim < min_similarity:
            logger.info(f"Rejected low-confidence correction (sim={sim:.2f}): "
                        f"'{text}' → '{out_text}'")
            return {
                "text": text,
                "corrected": False,
                "reason": "low_confidence",
                "confidence": round(sim, 3),
                "candidate": out_text,   # برای دیباگ
                "time_ms": int((time.time() - t0) * 1000),
            }

        return {
            "text": out_text,
            "corrected": out_text != text,
            "confidence": round(sim, 3),
            "time_ms": int((time.time() - t0) * 1000),
        }

    except Exception as e:
        logger.error(f"Inference failed: {e}", exc_info=True)
        return {"text": text, "corrected": False, "reason": f"inference_error: {e}",
                "confidence": 0.0, "time_ms": int((time.time() - t0) * 1000)}

# 🔴 سلول ۹ (جدید — جایگزین کن)


In [32]:
# ==============================================================
# Cell 9 (NEW) — Inference + Confidence Guard
# ==============================================================
from difflib import SequenceMatcher


def _similarity(a: str, b: str) -> float:
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return SequenceMatcher(None, a, b).ratio()


def _greedy_decode(model, src_ids, max_len=64):
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_mask = torch.ones(1, 1, src.size(1), dtype=torch.bool, device=device)
    with torch.no_grad():
        memory = model.encode(src, src_mask)
        ys = torch.tensor([[vocab.sos_idx]], dtype=torch.long, device=device)
        for _ in range(max_len):
            tgt_mask = ~torch.triu(
                torch.ones(1, ys.size(1), ys.size(1), dtype=torch.bool, device=device),
                diagonal=1,
            )
            dec_out = model.decode(memory, src_mask, ys, tgt_mask)
            logp = model.generator(dec_out[:, -1])        # ← generator، نه out
            next_id = int(logp.argmax(-1).item())
            ys = torch.cat([ys, torch.tensor([[next_id]], device=device)], dim=1)
            if next_id == vocab.eos_idx:
                break
    return ys[0].tolist()[1:-1]


def spell_correct(text: str, min_similarity: float = 0.75) -> dict:
    """
    Confidence guard: اگر خروجی مدل خیلی از ورودی فاصله گرفت، رد کن.
    دلیل: مدل روی ۴۰ جمله آموزش دیده و OOD را hallucinate می‌کند.
    """
    t0 = time.time()
    model = load_corrector()
    if model is None:
        return {"text": text, "corrected": False, "reason": "model_not_loaded",
                "confidence": 0.0, "time_ms": int((time.time() - t0) * 1000)}

    src_ids = vocab.encode(text[:64])
    if not src_ids:
        return {"text": text, "corrected": False, "reason": "empty_input",
                "confidence": 0.0, "time_ms": int((time.time() - t0) * 1000)}

    try:
        with torch.no_grad():
            out_ids = _greedy_decode(model, src_ids)
        out_text = vocab.decode(out_ids) or text

        sim = _similarity(text, out_text)
        if sim < min_similarity:
            logger.info(f"Rejected low-confidence (sim={sim:.2f}): "
                        f"'{text}' → '{out_text}'")
            return {
                "text": text,
                "corrected": False,
                "reason": "low_confidence",
                "confidence": round(sim, 3),
                "candidate": out_text,
                "time_ms": int((time.time() - t0) * 1000),
            }

        return {
            "text": out_text,
            "corrected": out_text != text,
            "confidence": round(sim, 3),
            "time_ms": int((time.time() - t0) * 1000),
        }
    except Exception as e:
        logger.error(f"Inference failed: {e}", exc_info=True)
        return {"text": text, "corrected": False, "reason": f"inference_error: {e}",
                "confidence": 0.0, "time_ms": int((time.time() - t0) * 1000)}


print("✅ Cell 9 loaded — spell_correct ready with Confidence Guard.")

✅ Cell 9 loaded — spell_correct ready with Confidence Guard.


# دومین سلول 9 جدید                🛠️ راه‌حل فوری: Confidence Guard سخت‌گیرانه + Fallback
سلول ۹ را با این نسخه جایگزین کن (سه لایه گارد امنیتی):

In [37]:
# ==============================================================
# Cell 9 (NEW v2) — Multi-layer Confidence Guard
# ==============================================================
from difflib import SequenceMatcher


def _char_similarity(a: str, b: str) -> float:
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return SequenceMatcher(None, a, b).ratio()


def _word_overlap(a: str, b: str) -> float:
    """چه نسبتی از کلمات ورودی در خروجی هم هستند."""
    a_words = set(a.split())
    b_words = set(b.split())
    if not a_words: return 1.0
    return len(a_words & b_words) / len(a_words)


def _greedy_decode(model, src_ids, max_len=64):
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_mask = torch.ones(1, 1, src.size(1), dtype=torch.bool, device=device)
    with torch.no_grad():
        memory = model.encode(src, src_mask)
        ys = torch.tensor([[vocab.sos_idx]], dtype=torch.long, device=device)
        for _ in range(max_len):
            tgt_mask = ~torch.triu(
                torch.ones(1, ys.size(1), ys.size(1), dtype=torch.bool, device=device),
                diagonal=1,
            )
            dec_out = model.decode(memory, src_mask, ys, tgt_mask)
            logp = model.generator(dec_out[:, -1])
            next_id = int(logp.argmax(-1).item())
            ys = torch.cat([ys, torch.tensor([[next_id]], device=device)], dim=1)
            if next_id == vocab.eos_idx:
                break
    return ys[0].tolist()[1:-1]


def spell_correct(
    text: str,
    min_char_sim: float = 0.90,    # ← از 0.75 به 0.90
    min_word_overlap: float = 0.70, # ← گارد جدید
    min_len_ratio: float = 0.75,    # ← گارد جدید
) -> dict:
    """
    سه گارد:
    1. char_similarity >= 0.90  (شباهت کاراکتری بالا)
    2. word_overlap    >= 0.70  (حداقل 70% کلمات حفظ شوند)
    3. length_ratio    بین 0.75 و 1.33 (خروجی خیلی کوتاه/بلند نباشد)
    """
    t0 = time.time()
    model = load_corrector()
    if model is None:
        return {"text": text, "corrected": False, "reason": "model_not_loaded",
                "confidence": 0.0, "time_ms": int((time.time() - t0) * 1000)}

    src_ids = vocab.encode(text[:64])
    if not src_ids:
        return {"text": text, "corrected": False, "reason": "empty_input",
                "confidence": 0.0, "time_ms": int((time.time() - t0) * 1000)}

    try:
        with torch.no_grad():
            out_ids = _greedy_decode(model, src_ids)
        out_text = vocab.decode(out_ids) or text

        # --- گارد ۱: char similarity ---
        char_sim = _char_similarity(text, out_text)
        if char_sim < min_char_sim:
            return _reject(text, out_text, "low_char_similarity", char_sim, t0)

        # --- گارد ۲: word overlap ---
        word_ov = _word_overlap(text, out_text)
        if word_ov < min_word_overlap:
            return _reject(text, out_text, "low_word_overlap", word_ov, t0)

        # --- گارد ۳: length ratio ---
        if len(text) > 0:
            ratio = len(out_text) / len(text)
            if ratio < min_len_ratio or ratio > 1.33:
                return _reject(text, out_text, "bad_length_ratio", ratio, t0)

        return {
            "text": out_text,
            "corrected": out_text != text,
            "confidence": round(char_sim, 3),
            "word_overlap": round(word_ov, 3),
            "time_ms": int((time.time() - t0) * 1000),
        }
    except Exception as e:
        logger.error(f"Inference failed: {e}", exc_info=True)
        return {"text": text, "corrected": False,
                "reason": f"inference_error: {e}", "confidence": 0.0,
                "time_ms": int((time.time() - t0) * 1000)}


def _reject(original, candidate, reason, score, t0):
    logger.info(f"Rejected [{reason}, score={score:.2f}]: "
                f"'{original}' → '{candidate}'")
    return {
        "text": original,
        "corrected": False,
        "reason": reason,
        "confidence": round(score, 3),
        "candidate": candidate,
        "time_ms": int((time.time() - t0) * 1000),
    }


print("✅ Cell 9 v2 loaded — strict guard (sim>=0.90, overlap>=0.70, len_ratio 0.75-1.33)")

✅ Cell 9 v2 loaded — strict guard (sim>=0.90, overlap>=0.70, len_ratio 0.75-1.33)


# سلول 10 — Pydantic Models (Type-safe API)


In [10]:
class SanitizeOptions(BaseModel):
    apply_zwnj: bool = True
    apply_digits_normalization: bool = True
    strip_urls: bool = True
    strip_html: bool = True
    collapse_repeats: bool = True


class TextIn(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)
    options: SanitizeOptions = Field(default_factory=SanitizeOptions)

    @field_validator("text")
    @classmethod
    def _no_null_bytes(cls, v: str) -> str:
        if "\x00" in v:
            raise ValueError("Null byte not allowed")
        return v


class TextOut(BaseModel):
    text: str
    changed: bool
    steps_applied: list[str]
    time_ms: int


class SanitizeCorrectOut(BaseModel):
    original: str
    sanitized: str
    corrected: str
    sanitizer_steps: list[str]
    spell_applied: bool
    total_time_ms: int


class HealthOut(BaseModel):
    status: str
    device: str
    model_loaded: bool
    vocab_size: int

# سلول 11 — FastAPI App + Endpoints


In [38]:
app = FastAPI(
    title="Persian Text Sanitizer + Spell Corrector API",
    version="1.0.0",
    description="Sanitize and correct Persian text. Built on Colab T4.",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)
app.add_middleware(GZipMiddleware, minimum_size=500)


@app.middleware("http")
async def add_timing_header(request: Request, call_next):
    t0 = time.time()
    response = await call_next(request)
    response.headers["X-Process-Time-Ms"] = str(int((time.time() - t0) * 1000))
    return response


@app.get("/", tags=["meta"])
def root():
    return {
        "name": "persian-text-sanitizer",
        "endpoints": ["/health", "/sanitize", "/correct", "/sanitize-correct", "/docs"],
    }


@app.get("/health", response_model=HealthOut, tags=["meta"])
def health():
    return HealthOut(
        status="ok",
        device=device.type,
        model_loaded=_corrector_model is not None,
        vocab_size=vocab.size,
    )


@app.post("/sanitize", response_model=TextOut, tags=["sanitize"])
def api_sanitize(body: TextIn, request: Request):
    t0 = time.time()
    result = sanitize(body.text, SanitizerConfig(**body.options.model_dump()))
    return TextOut(
        text=result["text"],
        changed=result["changed"],
        steps_applied=result["steps_applied"],
        time_ms=int((time.time() - t0) * 1000),
    )


@app.post("/correct", tags=["spell"])
async def api_correct(body: TextIn, request: Request):
    import asyncio
    # Run model inference in threadpool so we don't block the event loop
    result = await asyncio.to_thread(spell_correct, body.text)
    return result


@app.post("/sanitize-correct", response_model=SanitizeCorrectOut, tags=["pipeline"])
async def api_sanitize_correct(body: TextIn, request: Request):
    import asyncio
    t0 = time.time()

    # Step 1: sanitize (sync, CPU)
    san = sanitize(body.text, SanitizerConfig(**body.options.model_dump()))

    # Step 2: spell correct (GPU, run in thread)
    corr = await asyncio.to_thread(spell_correct, san["text"])

    return SanitizeCorrectOut(
        original=body.text,
        sanitized=san["text"],
        corrected=corr["text"],
        sanitizer_steps=san["steps_applied"],
        spell_applied=corr.get("corrected", False),
        total_time_ms=int((time.time() - t0) * 1000),
    )


@app.on_event("startup")
async def _startup():
    logger.info(f"API starting | device={device} | vocab={vocab.size}")

/tmp/ipykernel_977/3877937318.py:83: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


# 🔧 سلول ۱۲ (نسخه نهایی و مقاوم)

سلول 12 — اجرای API با ngrok (بدون بلاک کردن Colab)


In [39]:
# ==============================================================
# Cell 12 (FINAL) — Run FastAPI + cloudflared tunnel (robust)
# ==============================================================
import asyncio, threading, time, socket, re, os, subprocess
import nest_asyncio, requests
nest_asyncio.apply()

# ---------------------------------------------------------------
# (0) پاک‌سازی: ngrok و cloudflared قبلی را ببند
# ---------------------------------------------------------------
try:
    from pyngrok import ngrok
    ngrok.kill()
    print("Previous ngrok sessions killed.")
except Exception:
    pass

try:
    subprocess.run(["pkill", "-f", "cloudflared"], check=False)
    print("Previous cloudflared killed.")
except Exception:
    pass

# ---------------------------------------------------------------
# (1) پورت آزاد پیدا کن
# ---------------------------------------------------------------
def _find_free_port(start=8000, end=8010):
    for p in range(start, end):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                s.bind(("0.0.0.0", p))
                return p
            except OSError:
                continue
    raise RuntimeError("No free port in 8000-8010")

PORT = _find_free_port()
print(f"Using port: {PORT}")

# ---------------------------------------------------------------
# (2) uvicorn را در thread جدا با event loop اختصاصی اجرا کن
# ---------------------------------------------------------------
def _serve():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    cfg = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="warning",
        loop="none",           # ← خودمان loop را مدیریت می‌کنیم
    )
    server = uvicorn.Server(cfg)
    server.install_signal_handlers = lambda: None   # ← برای thread غیر main
    loop.run_until_complete(server.serve())

threading.Thread(target=_serve, daemon=True).start()

# ---------------------------------------------------------------
# (3) صبر کن تا /health واقعاً جواب بدهد (حداکثر 15 ثانیه)
# ---------------------------------------------------------------
def _wait_server(port, timeout=15):
    for _ in range(timeout * 10):
        try:
            r = requests.get(f"http://127.0.0.1:{port}/health", timeout=1)
            if r.status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(0.1)
    return False

if not _wait_server(PORT):
    # لاگ uvicorn را نشان بده تا کاربر بفهمد چه شد
    print("\n❌ Uvicorn did not start in 15s")
    print("→ Check that Cell 11 (FastAPI app) ran without errors")
    raise RuntimeError("Uvicorn failed to start")

print("✅ Uvicorn is UP")

# ---------------------------------------------------------------
# (4) cloudflared را دانلود و اجرا کن (بدون توکن، HTTPS رایگان)
# ---------------------------------------------------------------
CF_BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(CF_BIN):
    print("Downloading cloudflared...")
    subprocess.run([
        "wget", "-q", "-c",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", CF_BIN,
    ], check=False)
    os.chmod(CF_BIN, 0o755)

CF_LOG = "/tmp/cf.log"
open(CF_LOG, "w").close()

subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open(CF_LOG, "w"),
    stderr=subprocess.STDOUT,
)

# ---------------------------------------------------------------
# (5) صبر کن تا URL عمومی در لاگ ظاهر شود (حداکثر 30 ثانیه)
# ---------------------------------------------------------------
PUBLIC_URL = None
for _ in range(60):
    time.sleep(0.5)
    try:
        with open(CF_LOG) as f:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", f.read())
        if m:
            PUBLIC_URL = m.group(0)
            break
    except Exception:
        pass

# ---------------------------------------------------------------
# (6) نتیجه نهایی + ذخیره در globals برای سلول‌های بعدی
# ---------------------------------------------------------------
if PUBLIC_URL:
    globals()["PUBLIC_URL"] = PUBLIC_URL
    globals()["API_PORT"]   = PORT
    print("=" * 65)
    print(f"🌐 Public URL : {PUBLIC_URL}")
    print(f"📖 Docs       : {PUBLIC_URL}/docs")
    print(f"❤️  Health     : {PUBLIC_URL}/health")
    print("=" * 65)
    print("👉 این URL تا زمانی که runtime زنده است کار می‌کند.")
    print("👉 بعد از هر restart، این سلول را دوباره اجرا کن (URL عوض می‌شود).")
else:
    print("⚠️ Tunnel URL not found after 30s. Last log lines:")
    try:
        print(open(CF_LOG).read()[-1500:])
    except Exception:
        pass
    print(f"\nسرویس فقط داخل Colab در دسترس است: http://127.0.0.1:{PORT}")

Previous ngrok sessions killed.
Previous cloudflared killed.
Using port: 8003
✅ Uvicorn is UP
🌐 Public URL : https://hold-carrier-adjust-hung.trycloudflare.com
📖 Docs       : https://hold-carrier-adjust-hung.trycloudflare.com/docs
❤️  Health     : https://hold-carrier-adjust-hung.trycloudflare.com/health
👉 این URL تا زمانی که runtime زنده است کار می‌کند.
👉 بعد از هر restart، این سلول را دوباره اجرا کن (URL عوض می‌شود).


# 📋 سلول تست (کپی کن و اجرا کن)


In [40]:
# Test after strict guard
import requests, json
BASE = PUBLIC_URL

cases = [
    "من به مدرصه می‌روم",           # باید reject شود
    "برف روی کوه‌ها نشستة است",      # باید reject شود (مدل آشغال می‌دهد)
    "می‌روم به مدرصه",              # باید reject شود
    "سلام حال شما چطور است",         # شاید تصحیح شود (جمله آموزشی)
]

for t in cases:
    r = requests.post(f"{BASE}/correct", json={"text": t}, timeout=30).json()
    print(f"\nIN : {t}")
    print(f"OUT: {r.get('text')}")
    print(f"    corrected={r.get('corrected')} conf={r.get('confidence')} "
          f"reason={r.get('reason','-')}")
    if "candidate" in r:
        print(f"    rejected candidate: {r['candidate']}")


IN : من به مدرصه می‌روم
OUT: من به مدرصه می‌روم
    corrected=False conf=0.75 reason=low_char_similarity
    rejected candidate: من منده می‌ر م

IN : برف روی کوه‌ها نشستة است
OUT: برف روی کوه‌ها نشستة است
    corrected=False conf=0.56 reason=low_char_similarity
    rejected candidate: برو روی استه استه استه است

IN : می‌روم به مدرصه
OUT: می‌روم به مدرصه
    corrected=False conf=0.444 reason=low_char_similarity
    rejected candidate: موز می‌رودهم

IN : سلام حال شما چطور است
OUT: سلام حال شما چطور است
    corrected=False conf=0.686 reason=low_char_similarity
    rejected candidate: سلا حا حا حاست
